# Batch RGB vegetation-index extraction

## In short

This notebook calculates vegetation indices for each plot in a field trial, for each flight date.

You give it:

- One RGB orthomosaic (a stitched drone image) for each date.
- One plot grid shapefile (the plot boundaries) for each date.

You get one table (CSV file) for each date. Each row is one plot. The columns show the mean red, green, and blue values and 25 vegetation indices for that plot.

To use it, set two folder paths at the end of the code cell. Then run the cell.

## Details

**Input folder layout**

```
site_folder/
  RGB/
    YYYY-MM-DD/
      <orthomosaic>.tif      band 1 = Red, band 2 = Green, band 3 = Blue
      <plot_grid>.shp        can be in a subfolder
```

**Output**

- One CSV for each date: `output_root/<site>/<site>_<YYYY-MM-DD>_RGB_VI.csv`. `<site>` is the name of `site_folder`.
- `<site>_RGB_VI_batch_summary.csv`: the status, the raster, and the shapefile for each date.

**How to run**

1. In the `USER SETTINGS` block at the end of the code cell, set `site_folder` and `output_root`.
2. To process only some dates, set `dates_to_process` (for example `["2024-11-20"]`). This step is optional.
3. Run the cell.
4. Open the summary CSV. Look for dates with a status other than `processed`.

**Notes**

- The formulas and the masking are the same as in `Original_Template_Script/Generate_RGB indices_yam.ipynb`. This notebook does not need that file to run.
- A date folder can contain more than one `.tif` or `.shp`. In that case, `tif_sort_key` and `shapefile_sort_key` select one file. The summary CSV shows the files that the notebook used.
- The plot mean uses `rasterio.mask(..., filled=True, nodata=0)` and `np.nanmean`. Pixels inside the plot bounding box but outside the polygon get the value 0. The mean includes these zeros.


In [ ]:
from __future__ import annotations

import math
import re
from pathlib import Path
from typing import Iterable, Optional, Union

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask as raster_mask

try:
    from tqdm.auto import tqdm
except ImportError:  # pragma: no cover - only used if tqdm is unavailable
    def tqdm(iterable, **kwargs):
        return iterable


DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")
EXTRACTION_LOGIC = "original_notebook_filled_zero_mask"
DEFAULT_PLOT_ID_CANDIDATES = (
    "var",
    "id",
    "plot_id",
    "Plot_ID",
    "plot",
    "Plot",
    "name",
    "Name",
)

INDEX_COLUMNS = [
    "Mean Blue",
    "Mean Green",
    "Mean Red",
    "RCC",
    "GCC",
    "ExR",
    "ExG",
    "ExGR",
    "VARI_2",
    "MGRVI",
    "CIVE",
    "VEG",
    "RGVI",
    "NPCI",
    "SRPI",
    "NGRDI",
    "SCI",
    "GLI",
    "HI",
    "VARI",
    "GLAI",
    "CI",
    "SHP",
    "SAT",
    "HUE",
    "RI",
    "BI",
    "VVI",
]


def safe_name(value, keep_spaces: bool = False, max_len: int = 120) -> str:
    """Return a filesystem-safe name while keeping labels readable."""
    text = str(value).strip()

    if keep_spaces:
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r'[\\/:*?"<>|]+', "", text)
    else:
        text = text.replace(" ", "_")
        text = re.sub(r"[^A-Za-z0-9_\-()]+", "", text)

    text = text.strip(" ._")
    return (text[:max_len] if len(text) > max_len else text) or "unknown"


def find_orthophotos_root(path: Union[Path, str]) -> Path:
    """Find the Orthophotos folder from a site/script/notebook path."""
    path = Path(path).expanduser()
    try:
        resolved = path.resolve()
    except FileNotFoundError:
        resolved = path.absolute()

    for candidate in [resolved, *resolved.parents]:
        if candidate.name == "Orthophotos":
            return candidate

    for candidate in [resolved, *resolved.parents]:
        if (candidate / "Photos").exists() and (candidate / "Code").exists():
            return candidate

    return Path.cwd().resolve()


def find_rgb_folder(site_folder: Path) -> Path:
    """Find the RGB folder for a site."""
    direct = site_folder / "RGB"
    if direct.exists() and direct.is_dir():
        return direct

    candidates = sorted(
        p for p in site_folder.rglob("RGB")
        if p.is_dir() and "Clipped" not in p.parts
    )

    if not candidates:
        raise FileNotFoundError(f"No RGB folder found under: {site_folder}")

    if len(candidates) > 1:
        preview = "\n".join(f"  {p}" for p in candidates[:10])
        raise ValueError(
            "More than one RGB folder was found. Please set SITE_FOLDER to one site folder.\n"
            f"Candidates:\n{preview}"
        )

    return candidates[0]


def collect_date_folders(
    rgb_folder: Path,
    dates_to_process: Optional[Iterable[str]] = None,
) -> dict[str, Path]:
    date_folders = {
        p.name: p
        for p in rgb_folder.iterdir()
        if p.is_dir() and DATE_RE.match(p.name)
    }

    if dates_to_process:
        requested = set(dates_to_process)
        date_folders = {d: p for d, p in date_folders.items() if d in requested}

    return dict(sorted(date_folders.items()))


def tif_sort_key(path: Path):
    """Rank .tif files in a date folder. The lowest score is used."""
    name = path.name.lower()
    score = 100

    if "rgb" in name:
        score -= 30
    if re.search(r"(^|[-_])12($|[-_.])", name):
        score -= 20
    if "-12" in name or "_12" in name:
        score -= 10
    if "mns" in name:
        score += 10
    if "odm_orthophoto" in name or "original" in name:
        score += 40
    if "alias" in name:
        score += 100

    return score, name


def find_rgb_tif_for_date(date_dir: Path) -> Optional[Path]:
    tif_files = sorted(
        (
            p for p in date_dir.iterdir()
            if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
        ),
        key=tif_sort_key,
    )
    return tif_files[0] if tif_files else None


def shapefile_sort_key(path: Path, date_dir: Path):
    """Rank .shp files. Paths with "grid" are preferred, paths with "bad" are last."""
    relative = str(path.relative_to(date_dir)).lower()
    score = 0

    if "grid" not in relative:
        score += 30
    if "bad" in relative:
        score += 100

    return score, len(path.parts), relative


def shapefile_has_field(path: Path, field: str) -> bool:
    try:
        columns = gpd.read_file(path, rows=1).columns
    except TypeError:
        columns = gpd.read_file(path).columns
    except Exception:
        return False

    return field in columns


def find_grid_shapefile_for_date(
    date_dir: Path,
    plot_id_field: Optional[str] = None,
) -> Optional[Path]:
    candidates = sorted(
        (p for p in date_dir.rglob("*.shp") if not p.name.startswith(".")),
        key=lambda p: shapefile_sort_key(p, date_dir),
    )

    if not candidates:
        return None

    if plot_id_field:
        for candidate in candidates:
            if shapefile_has_field(candidate, plot_id_field):
                return candidate

    return candidates[0]


def resolve_plot_id_field(
    gdf: gpd.GeoDataFrame,
    requested_field: Optional[str] = None,
    candidates: Iterable[str] = DEFAULT_PLOT_ID_CANDIDATES,
) -> str:
    if requested_field:
        if requested_field not in gdf.columns:
            raise ValueError(
                f"Plot ID field '{requested_field}' not found. "
                f"Available fields: {list(gdf.columns)}"
            )
        return requested_field

    for candidate in candidates:
        if candidate in gdf.columns:
            return candidate

    geometry_name = gdf.geometry.name
    non_geometry_columns = [c for c in gdf.columns if c != geometry_name]
    if non_geometry_columns:
        return non_geometry_columns[0]

    raise ValueError("No non-geometry column was found for plot IDs.")


def safe_divide(numerator, denominator) -> float:
    try:
        denominator = float(denominator)
        numerator = float(numerator)
    except (TypeError, ValueError):
        return np.nan

    if not np.isfinite(denominator) or np.isclose(denominator, 0.0):
        return np.nan

    return numerator / denominator


def safe_power(value, power) -> float:
    with np.errstate(invalid="ignore", divide="ignore", over="ignore"):
        result = np.power(value, power)
    return float(result) if np.isfinite(result) else np.nan


def masked_band_mean(band) -> float:
    if np.ma.isMaskedArray(band):
        if band.count() == 0:
            return np.nan
        return float(np.ma.mean(band.astype("float64")))

    values = np.asarray(band, dtype="float64")
    if np.all(np.isnan(values)):
        return np.nan
    return float(np.nanmean(values))


def calculate_rgb_indices(red_mean: float, green_mean: float, blue_mean: float) -> dict[str, float]:
    rgb_sum = red_mean + green_mean + blue_mean
    red_green_sum = red_mean + green_mean
    green_red_minus_blue = green_mean + red_mean - blue_mean
    green_minus_blue = green_mean - blue_mean

    rgb_values = np.array([red_mean, green_mean, blue_mean], dtype="float64")
    if np.all(np.isnan(rgb_values)):
        rgb_max = np.nan
        rgb_min = np.nan
    else:
        rgb_max = float(np.nanmax(rgb_values))
        rgb_min = float(np.nanmin(rgb_values))

    exr = safe_divide(1.4 * red_mean - green_mean, rgb_sum)
    exg = safe_divide(2 * green_mean - red_mean - blue_mean, rgb_sum)

    veg_denominator = safe_power(red_mean, 0.667) * safe_power(blue_mean, 0.334)
    hue_input = safe_divide(2 * (red_mean - green_mean - blue_mean), 30.5)
    if np.isfinite(hue_input):
        hue_input = hue_input * green_minus_blue

    return {
        "Mean Blue": blue_mean,
        "Mean Green": green_mean,
        "Mean Red": red_mean,
        "RCC": safe_divide(red_mean, rgb_sum),
        "GCC": safe_divide(green_mean, rgb_sum),
        "ExR": exr,
        "ExG": exg,
        "ExGR": exg - exr if np.isfinite(exg) and np.isfinite(exr) else np.nan,
        "VARI_2": safe_divide(green_mean - red_mean, green_red_minus_blue),
        "MGRVI": safe_divide(green_mean ** 2 - red_mean ** 2, green_mean ** 2 + red_mean ** 2),
        "CIVE": (0.441 * red_mean) - (0.881 * green_mean) + (0.385 * blue_mean) + 18.787,
        "VEG": safe_divide(green_mean, veg_denominator),
        "RGVI": safe_divide(red_mean, green_mean),
        "NPCI": safe_divide(red_mean - blue_mean, red_mean + blue_mean),
        "SRPI": safe_divide(blue_mean, red_mean),
        "NGRDI": safe_divide(green_mean - red_mean, red_green_sum),
        "SCI": safe_divide(red_mean - green_mean, red_green_sum),
        "GLI": safe_divide(2 * green_mean - red_mean - blue_mean, 2 * green_mean + red_mean + blue_mean),
        "HI": safe_divide(2 * red_mean - green_mean - blue_mean, green_minus_blue),
        "VARI": safe_divide(green_mean - red_mean, rgb_sum),
        "GLAI": 25 * safe_divide(green_mean - red_mean, green_red_minus_blue) + 1.25,
        "CI": safe_divide(red_mean - blue_mean, red_mean),
        "SHP": 2 * safe_divide(red_mean - green_mean - blue_mean, green_minus_blue),
        "SAT": safe_divide(rgb_max - rgb_min, rgb_max),
        "HUE": math.atan(hue_input) if np.isfinite(hue_input) else np.nan,
        "RI": safe_divide(red_mean ** 2, blue_mean * green_mean ** 3),
        "BI": math.sqrt((red_mean ** 2 + green_mean ** 2 + blue_mean ** 2) / 3),
        "VVI": (
            (1 - safe_divide(red_mean - 30, red_mean + 30))
            * (1 - safe_divide(green_mean - 50, green_mean + 50))
            * (1 - safe_divide(blue_mean - 1, blue_mean + 1))
        ),
    }


def extract_rgb_indices_for_geometry(src, geometry, all_touched: bool = False) -> dict[str, float]:
    # Match the original Generate_RGB indices_yam.ipynb behavior exactly:
    # filled=True with nodata=0 keeps zero-filled pixels outside the polygon
    # inside the cropped window, and those zeros are included in np.nanmean.
    masked_image, _ = raster_mask(
        dataset=src,
        shapes=[geometry],
        nodata=0,
        filled=True,
        crop=True,
        all_touched=all_touched,
    )

    if masked_image.shape[0] < 3:
        raise ValueError(f"Expected at least 3 raster bands, found {masked_image.shape[0]}")

    red_mean = float(np.nanmean(masked_image[0, :, :]))
    green_mean = float(np.nanmean(masked_image[1, :, :]))
    blue_mean = float(np.nanmean(masked_image[2, :, :]))

    return calculate_rgb_indices(red_mean, green_mean, blue_mean)


def insert_metadata_columns(df: pd.DataFrame, site_name: str, date_str: str) -> pd.DataFrame:
    out = df.copy()

    site_col = "site" if "site" not in out.columns else "batch_site"
    date_col = "date" if "date" not in out.columns else "batch_date"

    out.insert(0, date_col, date_str)
    out.insert(0, site_col, site_name)
    return out


def extract_indices_for_date(
    raster_path: Path,
    shapefile_path: Path,
    output_csv: Path,
    site_name: str,
    date_str: str,
    plot_id_field: Optional[str] = None,
    all_touched: bool = False,
    overwrite: bool = True,
    include_metadata_columns: bool = False,
) -> dict:
    if output_csv.exists() and not overwrite:
        return {
            "status": "skipped_existing_output",
            "date": date_str,
            "features": 0,
            "warnings": 0,
            "plot_id_field": "",
            "output_csv": str(output_csv),
        }

    plots = gpd.read_file(shapefile_path)
    if plots.empty:
        raise ValueError(f"Shapefile has no features: {shapefile_path}")

    plots = plots[plots.geometry.notnull()].copy()
    if plots.empty:
        raise ValueError(f"Shapefile has no valid geometries: {shapefile_path}")

    with rasterio.open(raster_path) as src:
        if src.count < 3:
            raise ValueError(f"Expected at least 3 raster bands, found {src.count}: {raster_path}")

        if plots.crs and src.crs and plots.crs != src.crs:
            plots = plots.to_crs(src.crs)
        elif plots.crs is None:
            print(f"[WARNING] Shapefile CRS is missing: {shapefile_path}")
        elif src.crs is None:
            print(f"[WARNING] Raster CRS is missing: {raster_path}")

        resolved_plot_id_field = resolve_plot_id_field(plots, plot_id_field)

        records = []
        warnings_count = 0

        iterator = tqdm(
            plots.iterrows(),
            total=len(plots),
            desc=f"Extracting {date_str}",
        )

        for _, row in iterator:
            try:
                indices = extract_rgb_indices_for_geometry(
                    src,
                    row.geometry,
                    all_touched=all_touched,
                )
            except Exception as exc:
                warnings_count += 1
                print(f"[WARNING] {date_str} plot {row[resolved_plot_id_field]} failed: {exc}")
                indices = {column: np.nan for column in INDEX_COLUMNS}

            records.append(indices)

    indices_df = pd.DataFrame(records, columns=INDEX_COLUMNS)
    result = pd.concat(
        [plots.reset_index(drop=True), indices_df.reset_index(drop=True)],
        axis=1,
    )
    if include_metadata_columns:
        result = insert_metadata_columns(result, site_name=site_name, date_str=date_str)

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_csv, index=False)

    return {
        "status": "processed",
        "date": date_str,
        "features": len(result),
        "warnings": warnings_count,
        "plot_id_field": resolved_plot_id_field,
        "output_csv": str(output_csv),
    }


def run_batch_rgb_vi(
    site_folder: Union[Path, str],
    dates_to_process: Optional[Iterable[str]] = None,
    output_root: Optional[Union[Path, str]] = None,
    plot_id_field: Optional[str] = None,
    site_name_override: Optional[str] = None,
    overwrite: bool = True,
    all_touched: bool = False,
    include_metadata_columns: bool = False,
) -> pd.DataFrame:
    site_folder = Path(site_folder).expanduser().resolve()
    if not site_folder.exists():
        raise FileNotFoundError(f"Site folder does not exist: {site_folder}")

    site_name = safe_name(site_name_override or site_folder.name, keep_spaces=True)
    rgb_folder = find_rgb_folder(site_folder)

    if output_root is None:
        output_root = find_orthophotos_root(site_folder) / "extracted VI"
    else:
        output_root = Path(output_root).expanduser().resolve()

    output_root.mkdir(parents=True, exist_ok=True)

    # All CSVs for one site go into one folder:
    # <output_root>/<site>/<site>_<YYYY-MM-DD>_RGB_VI.csv
    site_output_dir = output_root / site_name
    site_output_dir.mkdir(parents=True, exist_ok=True)

    date_folders = collect_date_folders(rgb_folder, dates_to_process)
    if not date_folders:
        print(f"No date folders found under: {rgb_folder}")
        return pd.DataFrame()

    print("Starting batch RGB VI extraction")
    print(f"Extraction logic: {EXTRACTION_LOGIC}")
    print(f"Site folder: {site_folder}")
    print(f"RGB folder: {rgb_folder}")
    print(f"Output root: {output_root}")
    print(f"Site output folder: {site_output_dir}")
    print(f"Dates selected: {len(date_folders)}")

    summary_rows = []

    for date_str, date_dir in date_folders.items():
        print("\n" + "=" * 70)
        print(f"Processing {site_name} {date_str}")

        raster_path = find_rgb_tif_for_date(date_dir)
        shapefile_path = find_grid_shapefile_for_date(date_dir, plot_id_field=plot_id_field)

        output_csv = site_output_dir / f"{site_name}_{date_str}_RGB_VI.csv"

        row = {
            "site": site_name,
            "date": date_str,
            "date_folder": str(date_dir),
            "raster": str(raster_path) if raster_path else "",
            "shapefile": str(shapefile_path) if shapefile_path else "",
            "output_csv": str(output_csv),
            "status": "",
            "features": 0,
            "warnings": 0,
            "plot_id_field": "",
            "message": "",
        }

        if raster_path is None:
            row["status"] = "skipped_missing_rgb_tif"
            row["message"] = "No RGB .tif file found directly inside the date folder."
            print(f"[SKIP] {row['message']}")
            summary_rows.append(row)
            continue

        if shapefile_path is None:
            row["status"] = "skipped_missing_shapefile"
            row["message"] = "No .shp file found inside the date folder."
            print(f"[SKIP] {row['message']}")
            summary_rows.append(row)
            continue

        print(f"RGB tif: {raster_path}")
        print(f"Shapefile: {shapefile_path}")
        print(f"Output CSV: {output_csv}")

        try:
            stats = extract_indices_for_date(
                raster_path=raster_path,
                shapefile_path=shapefile_path,
                output_csv=output_csv,
                site_name=site_name,
                date_str=date_str,
                plot_id_field=plot_id_field,
                all_touched=all_touched,
                overwrite=overwrite,
                include_metadata_columns=include_metadata_columns,
            )
            row.update(stats)
            if stats["status"] == "processed":
                row["message"] = "OK"
                print(f"[DONE] Wrote {stats['features']} rows")
            else:
                row["message"] = "Existing output kept."
                print(f"[SKIP] {row['message']}")
        except Exception as exc:
            row["status"] = "failed"
            row["message"] = str(exc)
            print(f"[FAILED] {exc}")

        summary_rows.append(row)

    summary = pd.DataFrame(summary_rows)
    summary_csv = site_output_dir / f"{site_name}_RGB_VI_batch_summary.csv"
    summary.to_csv(summary_csv, index=False)

    print("\n" + "#" * 70)
    print("FINAL RUN SUMMARY")
    print("#" * 70)
    print(summary["status"].value_counts(dropna=False).to_string())
    print(f"\nSummary CSV: {summary_csv}")

    return summary


# ============================================================
# USER SETTINGS
# Edit only this section when switching sites.
# ============================================================

if __name__ == "__main__":
    # Folder that contains RGB/YYYY-MM-DD/ date folders.
    site_folder = 'path/to/site_folder'

    # CSVs are written to a subfolder named after the site folder.
    output_root = 'path/to/output'

    # None processes every date folder. Or give a list, e.g. ["2024-11-20"].
    dates_to_process = None

    summary = run_batch_rgb_vi(
        site_folder=site_folder,
        dates_to_process=dates_to_process,
        output_root=output_root,
    )
    display(summary)
